# 14 Pi FP32 Runtime Diagnosis

이 노트북은 10번의 Pi offline runtime 결과, 13번의 실제 주행 로그, 그리고 14번 `pi_runtime_probe`에서 새로 가져온 telemetry 로그를 비교한다.

핵심 질문은 모델 정확도가 아니라 **FP32 lane stack이 실제 시험 최소 구성에서 Pi 보드 위에서 안정적으로 버티는가**이다.

## 해석 기준

- `pipeline_ms`: 한 frame 전체 처리 시간. `fps = 1000 / pipeline_ms`.
- `inference_ms`: ONNX 모델 추론 시간. 현재 병목의 대부분이다.
- `throttled`: `vcgencmd get_throttled` 결과. `throttled=0x0`이면 정상.
- `temp_c`, `arm_clock_mhz`: 발열로 clock이 떨어지는지 확인한다.

14번 probe는 overlay, 이미지 저장, 동영상 저장, Jupyter를 모두 제거한 minimal runtime이다.

In [ ]:
from pathlib import Path
import json
import pandas as pd

EXP_ROOT = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments")
EXP12 = EXP_ROOT / "12_clrkdnet_supervised_rebuild"
EXP13 = EXP_ROOT / "13_pi_drive_test_fp32_clrkdnet"
EXP14 = EXP_ROOT / "14_pi_fp32_runtime_diagnosis"

PI10 = EXP12 / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg_from_pi" / "out_pi" / "runtime_validation_report.json"
LOCAL10 = EXP12 / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg" / "out_local" / "runtime_validation_report.json"
RUN13 = EXP13 / "02_extracted" / "drive_test" / "drive_runs"
RESULTS14 = EXP14 / "results_from_pi"

print('PI10 exists:', PI10.exists())
print('LOCAL10 exists:', LOCAL10.exists())
print('RUN13 exists:', RUN13.exists())
print('RESULTS14 exists:', RESULTS14.exists())

In [ ]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def report_latency_row(label, report):
    lat = report['latency_ms']
    return {
        'source': label,
        'machine': report['environment'].get('machine', ''),
        'onnxruntime': report['environment'].get('onnxruntime', ''),
        'threads': report['environment'].get('pi_ort_threads', ''),
        'inference_mean_ms': lat['inference']['mean'],
        'pipeline_mean_ms': lat['pipeline_no_disk']['mean'],
        'pipeline_p95_ms': lat['pipeline_no_disk']['p95'],
        'fps_mean': lat['fps_no_disk_mean'],
    }

rows = []
if LOCAL10.exists():
    rows.append(report_latency_row('10 local offline', load_json(LOCAL10)))
if PI10.exists():
    rows.append(report_latency_row('10 pi offline', load_json(PI10)))
pd.DataFrame(rows)

In [ ]:
def summarize_drive_log(path):
    df = pd.read_csv(path)
    out = {
        'run': path.parent.name,
        'frames': len(df),
        'pipeline_mean_ms': df['pipeline_ms'].mean(),
        'pipeline_p95_ms': df['pipeline_ms'].quantile(0.95),
        'pipeline_max_ms': df['pipeline_ms'].max(),
        'inference_mean_ms': df['inference_ms'].mean(),
        'inference_p95_ms': df['inference_ms'].quantile(0.95),
    }
    if 'fps_est' in df.columns:
        out['fps_est_mean'] = df['fps_est'].mean()
    else:
        out['fps_est_mean'] = 1000.0 / out['pipeline_mean_ms']
    return out

run_rows = []
if RUN13.exists():
    for log in sorted(RUN13.glob('*/drive_log.csv')):
        run_rows.append(summarize_drive_log(log))
df13 = pd.DataFrame(run_rows)
df13

In [ ]:
def summarize_probe_log(path):
    df = pd.read_csv(path)
    out = {
        'run': path.parent.name,
        'frames': len(df),
        'pipeline_mean_ms': df['pipeline_ms'].mean(),
        'pipeline_p95_ms': df['pipeline_ms'].quantile(0.95),
        'pipeline_max_ms': df['pipeline_ms'].max(),
        'inference_mean_ms': df['inference_ms'].mean(),
        'inference_p95_ms': df['inference_ms'].quantile(0.95),
        'fps_ema_last': df['fps_ema'].iloc[-1] if len(df) else None,
        'temp_max_c': pd.to_numeric(df.get('temp_c'), errors='coerce').max() if 'temp_c' in df else None,
        'clock_min_mhz': pd.to_numeric(df.get('arm_clock_mhz'), errors='coerce').min() if 'arm_clock_mhz' in df else None,
        'throttled_values': ', '.join(sorted(set(str(x) for x in df.get('throttled', pd.Series(dtype=str)).dropna()))),
    }
    return out

probe_rows = []
if RESULTS14.exists():
    for log in sorted(RESULTS14.glob('**/runtime_log.csv')):
        probe_rows.append(summarize_probe_log(log))
df14 = pd.DataFrame(probe_rows)
df14

## 다음 해석

1. `10 pi offline`과 `13 live drive`가 비슷하면, 기본 병목은 ONNX inference다.
2. `14 minimal drive`가 13보다 뚜렷하게 안정적이면, display/save/print 오버헤드가 컸던 것이다.
3. `14 minimal drive`에서도 latency가 증가하거나 `throttled != 0x0`이면, 전원/발열/CPU 한계가 우선 문제다.
4. 이 결과가 안정적일 때만 sign/traffic/red line 이벤트를 얹는다.